# Exploration 01 — OptionMetrics `vsurfd` (volatility surface)

Objective: locate the vol surface table, read its **actual columns**, confirm that **91 days / delta 50** exist, then (next step) find the SPX `secid`.

Read-only — no heavy data downloaded (DISTINCT on low-cardinality columns).

In [1]:
from dispersion.data.wrds_client import get_connection

db = get_connection()

Loading library list...


Done


## 1) Where does `vsurfd` live? (annual partitioning?)

In [2]:
for lib in ["optionm_all", "optionm"]:
    try:
        tables = db.list_tables(library=lib)
    except Exception as e:
        print(f"[{lib}] inaccessible: {e}")
        continue
    vsurf = sorted(t for t in tables if "vsurf" in t.lower())
    print(f"[{lib}] {len(tables)} tables total | vsurf*: {vsurf}")
# Result: annual tables vsurfd1996 -> vsurfd2025 (~30 years), in optionm and optionm_all.

[optionm_all] 402 tables total | vsurf*: ['vsurfbr1996', 'vsurfbr1997', 'vsurfbr1998', 'vsurfbr1999', 'vsurfbr2000', 'vsurfbr2001', 'vsurfbr2002', 'vsurfbr2003', 'vsurfbr2004', 'vsurfbr2005', 'vsurfbr2006', 'vsurfbr2007', 'vsurfbr2008', 'vsurfbr2009', 'vsurfbr2010', 'vsurfbr2011', 'vsurfbr2012', 'vsurfbr2013', 'vsurfbr2014', 'vsurfbr2015', 'vsurfbr2016', 'vsurfbr2017', 'vsurfbr2018', 'vsurfbr2019', 'vsurfbr2020', 'vsurfbr2021', 'vsurfbr2022', 'vsurfbr2023', 'vsurfbr2024', 'vsurfbr2025', 'vsurfd1996', 'vsurfd1997', 'vsurfd1998', 'vsurfd1999', 'vsurfd2000', 'vsurfd2001', 'vsurfd2002', 'vsurfd2003', 'vsurfd2004', 'vsurfd2005', 'vsurfd2006', 'vsurfd2007', 'vsurfd2008', 'vsurfd2009', 'vsurfd2010', 'vsurfd2011', 'vsurfd2012', 'vsurfd2013', 'vsurfd2014', 'vsurfd2015', 'vsurfd2016', 'vsurfd2017', 'vsurfd2018', 'vsurfd2019', 'vsurfd2020', 'vsurfd2021', 'vsurfd2022', 'vsurfd2023', 'vsurfd2024', 'vsurfd2025']


[optionm] 578 tables total | vsurf*: ['vsurfbr1996', 'vsurfbr1997', 'vsurfbr1998', 'vsurfbr1999', 'vsurfbr2000', 'vsurfbr2001', 'vsurfbr2002', 'vsurfbr2003', 'vsurfbr2004', 'vsurfbr2005', 'vsurfbr2006', 'vsurfbr2007', 'vsurfbr2008', 'vsurfbr2009', 'vsurfbr2010', 'vsurfbr2011', 'vsurfbr2012', 'vsurfbr2013', 'vsurfbr2014', 'vsurfbr2015', 'vsurfbr2016', 'vsurfbr2017', 'vsurfbr2018', 'vsurfbr2019', 'vsurfbr2020', 'vsurfbr2021', 'vsurfbr2022', 'vsurfbr2023', 'vsurfbr2024', 'vsurfbr2025', 'vsurfd1996', 'vsurfd1997', 'vsurfd1998', 'vsurfd1999', 'vsurfd2000', 'vsurfd2001', 'vsurfd2002', 'vsurfd2003', 'vsurfd2004', 'vsurfd2005', 'vsurfd2006', 'vsurfd2007', 'vsurfd2008', 'vsurfd2009', 'vsurfd2010', 'vsurfd2011', 'vsurfd2012', 'vsurfd2013', 'vsurfd2014', 'vsurfd2015', 'vsurfd2016', 'vsurfd2017', 'vsurfd2018', 'vsurfd2019', 'vsurfd2020', 'vsurfd2021', 'vsurfd2022', 'vsurfd2023', 'vsurfd2024', 'vsurfd2025']


## 2) Actual columns + available maturities/deltas (reference year 2024)

In [3]:
print('=== Columns of optionm.vsurfd2024 ===')
print(db.describe_table(library='optionm', table='vsurfd2024'))

print('\n=== Available maturities (days) ===')
print(db.raw_sql('SELECT DISTINCT days FROM optionm.vsurfd2024 ORDER BY days')['days'].tolist())

print('\n=== Available deltas ===')
print(db.raw_sql('SELECT DISTINCT delta FROM optionm.vsurfd2024 ORDER BY delta')['delta'].tolist())

print('\n=== Available cp_flag ===')
print(db.raw_sql('SELECT DISTINCT cp_flag FROM optionm.vsurfd2024')['cp_flag'].tolist())

# Key result:
#  columns = secid, date, days, delta, impl_volatility, impl_strike, impl_premium, dispersion, cp_flag
#  days includes 91 EXACTLY   -> our 3-month maturity is native
#  delta includes +/-50       -> ATM available (call delta=50, put delta=-50)
#  ~536 M rows / year -> FILTER AT THE SOURCE (days=91, delta in (50,-50), restricted secid)
#  NB: the 'dispersion' column here = OptionMetrics accuracy measure, NOTHING to do with our signal.

=== Columns of optionm.vsurfd2024 ===


Approximately 535788672 rows in optionm.vsurfd2024.


              name  nullable              type  \
0            secid      True  DOUBLE PRECISION   
1             date      True              DATE   
2             days      True  DOUBLE PRECISION   
3            delta      True  DOUBLE PRECISION   
4  impl_volatility      True  DOUBLE PRECISION   
5      impl_strike      True  DOUBLE PRECISION   
6     impl_premium      True  DOUBLE PRECISION   
7       dispersion      True  DOUBLE PRECISION   
8          cp_flag      True        VARCHAR(1)   

                                             comment  
0                                        Security ID  
1                                               Date  
2                                 Days to Expiration  
3                                Delta of the Option  
4      Interpolated Implied Volatility of the Option  
5       The Strike Price Corresponding to this Delta  
6  The Premium of a Theoretical Option with this ...  
7  A Measure of the Accuracy of the Implied Volat...  
8   

[10.0, 30.0, 60.0, 91.0, 122.0, 152.0, 182.0, 273.0, 365.0, 547.0, 730.0]

=== Available deltas ===


[-90.0, -85.0, -80.0, -75.0, -70.0, -65.0, -60.0, -55.0, -50.0, -45.0, -40.0, -35.0, -30.0, -25.0, -20.0, -15.0, -10.0, 10.0, 15.0, 20.0, 25.0, 30.0, 35.0, 40.0, 45.0, 50.0, 55.0, 60.0, 65.0, 70.0, 75.0, 80.0, 85.0, 90.0]

=== Available cp_flag ===


['C', 'P']
